In [9]:
# pip install yfinance pandas textblob python-dateutil requests tweepy

import os
import pandas as pd
from textblob import TextBlob
from datetime import datetime
from dateutil.tz import tzlocal
import yfinance as yf

# Optional Twitter API
try:
    import tweepy
    TWITTER_ENABLED = True
except ImportError:
    TWITTER_ENABLED = False

# ---------- CONFIG ----------
STOCK = "INFY.NS"
NEWS_PER_STOCK = 80
PRICE_PERIOD = "180d"
OUT_DIR = "outputs"
USE_TWITTER = False          # set True if Twitter API credentials available

# Twitter API credentials (v2)
TWITTER_BEARER_TOKEN = os.environ.get("TWITTER_BEARER_TOKEN")

os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Utilities ----------
def calc_sentiment(text: str):
    blob = TextBlob(text or "")
    polarity = blob.sentiment.polarity
    if polarity > 0.05:
        label = "Positive"
    elif polarity < -0.05:
        label = "Negative"
    else:
        label = "Neutral"
    return polarity, label

def fetch_news_sentiment(ticker: str, n: int = 80) -> pd.DataFrame:
    """Fetch Yahoo Finance news for ticker and compute sentiment."""
    tk = yf.Ticker(ticker)
    items = tk.news or []
    rows = []
    for item in items[:n]:
        title = item.get("title") or ""
        provider = item.get("providerPublishTime") or ""
        url = item.get("link") or ""
        pub = item.get("providerPublishTime")
        ts = pd.to_datetime(pub, unit="s", utc=True) if pub else pd.NaT
        score, label = calc_sentiment(title)
        rows.append({
            "stock": ticker,
            "title": title,
            "url": url,
            "provider": provider,
            "pubDate": ts,
            "sentiment_score": score,
            "sentiment_label": label
        })
    df = pd.DataFrame(rows)
    df = df.dropna(subset=["pubDate"])
    return df

def fetch_tweets_sentiment(query: str, max_tweets=500) -> pd.DataFrame:
    """Fetch tweets using Twitter API v2 and compute sentiment."""
    if not USE_TWITTER or not TWITTER_BEARER_TOKEN:
        return pd.DataFrame(columns=["date","content","user","sentiment_score","sentiment_label"])
    
    client = tweepy.Client(bearer_token=TWITTER_BEARER_TOKEN, wait_on_rate_limit=True)
    tweets_list = []
    
    for tweet in tweepy.Paginator(client.search_recent_tweets,
                                  query=query,
                                  tweet_fields=["created_at","text","author_id"],
                                  max_results=100).flatten(limit=max_tweets):
        score, label = calc_sentiment(tweet.text)
        tweets_list.append([tweet.created_at, tweet.text, tweet.author_id, score, label])
        
    df = pd.DataFrame(tweets_list, columns=["date","content","user","sentiment_score","sentiment_label"])
    return df

def build_sentiment_indices(df_news: pd.DataFrame):
    """Return weekly and monthly sentiment index."""
    df = df_news.copy()
    df["pubDate"] = pd.to_datetime(df["pubDate"], utc=True)
    
    weekly = (df.set_index("pubDate")
              .resample("W")["sentiment_score"]
              .mean()
              .reset_index()
              .rename(columns={"sentiment_score": "avg_sentiment_weekly"}))
    
    monthly = (df.set_index("pubDate")
               .resample("M")["sentiment_score"]
               .mean()
               .reset_index()
               .rename(columns={"sentiment_score": "avg_sentiment_monthly"}))
    
    return weekly, monthly

# ---------- Main ----------
def main():
    # 1) Fetch Yahoo Finance news
    news_df = fetch_news_sentiment(STOCK, n=NEWS_PER_STOCK)
    
    # 2) Fetch tweets if enabled
    if USE_TWITTER:
        query = "Infosys -is:retweet lang:en"
        tweets_df = fetch_tweets_sentiment(query)
        # unify column names
        tweets_df = tweets_df.rename(columns={"date":"pubDate"})
        combined_df = pd.concat([news_df, tweets_df], ignore_index=True)
    else:
        combined_df = news_df
    
    if combined_df.empty:
        print("No news/tweets found. Exiting.")
        return
    
    # 3) Build weekly/monthly sentiment
    weekly, monthly = build_sentiment_indices(combined_df)
    
    # 4) Save to CSV
    combined_df.to_csv(os.path.join(OUT_DIR, "infosys_news_tweets.csv"), index=False)
    weekly.to_csv(os.path.join(OUT_DIR, "infosys_weekly_sentiment.csv"), index=False)
    monthly.to_csv(os.path.join(OUT_DIR, "infosys_monthly_sentiment.csv"), index=False)
    
    # 5) Print snapshot
    print("Weekly sentiment snapshot:")
    print(weekly.tail(5))
    print("\nMonthly sentiment snapshot:")
    print(monthly.tail(5))
    print("\nDone. CSV files saved in", OUT_DIR)

if __name__ == "__main__":
    main()


No news/tweets found. Exiting.


In [10]:
import requests, pandas as pd

url = "https://newsapi.org/v2/everything"
params = {"q": "Infosys", "from": "2025-08-01", "sortBy": "publishedAt", "apiKey": "YOUR_API_KEY"}
r = requests.get(url, params=params).json()
articles = [[a["publishedAt"], a["title"]] for a in r["articles"]]
df_news = pd.DataFrame(articles, columns=["date", "headline"])
print(df_news.head())


KeyError: 'articles'